# Employee Salary Prediction using AdaBoost Regressor


## 1. Problem Statement
In modern organizations, determining fair and accurate employee salaries is critical for retention, motivation, and performance management. However, salary depends on multiple factors such as experience, education, performance, and work behavior.

This project aims to build a machine learning regression model that can predict an employee’s annual salary (in lakhs) based on various features like:

Experience
Performance score
Education level
Certifications
Work habits (overtime, remote work)

- The challenge is to capture non-linear relationships between these variables and salary.

## 2. Objectives
- Primary Objectives
   - Build a regression model to predict Annual Salary (Lakhs)
   - Apply AdaBoost Regressor for improved accuracy
   - Evaluate model using:
       - MAE (Mean Absolute Error)
       - R² Score
- Secondary Objectives
   - Perform EDA (Exploratory Data Analysis)
   - Handle categorical variables (encoding)
   - Compare performance with a baseline model (like Decision Tree)
   - Identify important features influencing salary

## 3.Import require libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 4.Load Dataset

In [6]:


df = pd.read_csv("adaboost_regression_dataset.csv")
df.head(9)


,Age,Experience_Years,Education_Level,Department,Num_Projects,Performance_Score,Certifications,Remote_Work,Overtime_Hours,Training_Hours,Annual_Salary_Lakhs
0,50,28,Bachelor,Finance,15,2.92,4,0,3,70,34.13
1,36,13,Bachelor,HR,12,3.67,4,0,10,69,26.91
2,29,6,Bachelor,HR,1,3.76,3,1,4,10,19.52
3,42,21,Bachelor,Engineering,13,3.28,4,0,16,81,30.03
4,40,16,Master,Data Science,14,3.85,2,1,19,92,32.86
5,44,21,Master,Marketing,8,2.24,5,0,6,88,30.35
6,32,10,Bachelor,Engineering,8,2.26,1,1,14,62,21.68
7,32,11,PhD,HR,12,2.52,4,0,14,20,29.54
8,45,21,PhD,Marketing,1,3.55,2,1,13,2,29.78


## 5.Check Missing Values & Handle

In [7]:
df.isnull().sum()

Age                    0
Experience_Years       0
Education_Level        0
Department             0
Num_Projects           0
Performance_Score      0
Certifications         0
Remote_Work            0
Overtime_Hours         0
Training_Hours         0
Annual_Salary_Lakhs    0
dtype: int64

## 6.Data Preprocessing

### Encode Categorical Variables

In [8]:
from sklearn.preprocessing import LabelEncoder

# Label Encoding
le = LabelEncoder()
df['Education_Level'] = le.fit_transform(df['Education_Level'])

# One-hot Encoding
df = pd.get_dummies(df, columns=['Department'], drop_first=True)

## 7.Model Building

### i) Seperate Target & Feature

In [9]:
X = df.drop('Annual_Salary_Lakhs', axis=1)
y = df['Annual_Salary_Lakhs']

### ii) Split The Dataset

In [10]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=42)

# Baseline Model (Decision Tree)

In [12]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score

dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)



DecisionTreeRegressor(random_state=42)

In [13]:
y_pred_dt = dt.predict(X_test)

print("Decision Tree Results:")
print("MAE:", mean_absolute_error(y_test, y_pred_dt))
print("R2 Score:", r2_score(y_test, y_pred_dt))

Decision Tree Results:
MAE: 2.3225247524752475
R2 Score: 0.8221278985555419


# AdaBoost Regressor

In [14]:
from sklearn.ensemble import AdaBoostRegressor

base_model = DecisionTreeRegressor(max_depth=3)

ada = AdaBoostRegressor(
    estimator=base_model,
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

ada.fit(X_train, y_train)

AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=3),
                  learning_rate=0.1, n_estimators=100, random_state=42)

In [15]:
y_pred_ada = ada.predict(X_test)

print("\nAdaBoost Results:")
print("MAE:", mean_absolute_error(y_test, y_pred_ada))
print("R2 Score:", r2_score(y_test, y_pred_ada))


AdaBoost Results:
MAE: 2.1158181090708394
R2 Score: 0.8551254552569791


## 8.Hyperparameter Tuning

In [16]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.5],
    'estimator__max_depth': [2, 3, 5]
}

grid = GridSearchCV(
    AdaBoostRegressor(estimator=DecisionTreeRegressor()),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

Best Parameters: {'estimator__max_depth': 5, 'learning_rate': 0.5, 'n_estimators': 200}


## 10. Final Evaluation After Tuning

In [17]:
best_model = grid.best_estimator_

y_pred_best = best_model.predict(X_test)

print("\nTuned AdaBoost Results:")
print("MAE:", mean_absolute_error(y_test, y_pred_best))
print("R2 Score:", r2_score(y_test, y_pred_best))


Tuned AdaBoost Results:
MAE: 1.5881543121130501
R2 Score: 0.9185318287131163


## 11.Conclusion

This project demonstrates the effectiveness of ensemble learning in solving real-world regression problems. Starting with a Decision Tree as a baseline, the model performance was significantly improved using AdaBoost, which iteratively focused on correcting errors made by previous models.

The results clearly show that hyperparameter tuning plays a crucial role, as the tuned AdaBoost model achieved the best performance with lowest MAE (1.58) and highest R² score (0.91).

Overall, this project highlights how boosting techniques can enhance prediction accuracy, handle complex data patterns, and outperform traditional models. It also reinforces the importance of model comparison, evaluation metrics, and optimization in building reliable machine learning solutions.